In [ ]:
import numpy as np
import pandas as pd
import csv
import sys
import os
from sklearn.decomposition import NMF

# Notebook uses functions from .py scripts in scripts/ folder
sys.path.append('../scripts/calculate')

from metrics import cosine_sim, get_evar

## Non-negative Matrix Factorization

This notebook performs Non-negative Matrix Factorization on a selected data type (genera/functions) using the training set, selects the best model based on validation set reconstruction, and finally reports evaluation metrics on the test set.

In [ ]:
TOPICS = [7]
exp_group = 'random_split'
data_type = 'functions' # choices: 'functions', 'genera'

In [5]:
# Load data
X_train = pd.read_csv(f'../data/{data_type}/{exp_group}/processed/train.tsv', index_col=0, sep="\t")
X_val = pd.read_csv(f'../data/{data_type}/{exp_group}/processed/val.tsv', index_col=0, sep="\t")
X_test = pd.read_csv(f'../data/{data_type}/{exp_group}/processed/test.tsv', index_col=0, sep="\t")

print((X_train.shape, X_val.shape, X_test.shape))

((7836, 623), (2612, 623), (2612, 623))


In [ ]:
# Save metrics
def save_metric(metrics, filename):
    with open (filename, 'w') as f:
        writer = csv.writer(f)

        for topic, value in zip(TOPICS, metrics):        
            writer.writerow([topic, value])
            
def save_metric_per_run(metrics, nruns, filename):
    nruns = [i for i in range(1, nruns+1)]
    with open (filename, 'w') as f:
        writer = csv.writer(f)
        for run, value in zip(nruns, metrics):        
            writer.writerow([run, value])

In [ ]:
def run_nmf(
    X_train: pd.DataFrame,
    X_val: pd.DataFrame,
    nsig: int,
    outdir:str,
    alpha: float,
    divergence: str = 'kl',
    solvernmf: str = "mu",
    l1ratio: float = 0.0,
    iternb: int = 2000,
    regu: str = 'both',
    nruns: int = 100,
    random_seed: int = 0
):
    """
    Run Non-negative Matrix Factorization (NMF) on an abundance matrix.

    Args:
        X_train: Input train abundance matrix (samples x features).
        nsig: Number of signatures.
        alpha: Regularization parameter.
        divergence: Divergence metric ('kl' or 'frobenius'). Defaults to 'kl'.
        solvernmf: Solver for NMF ('mu' or 'cd'). Defaults to 'mu'.
        l1ratio: L1/L2 regularization ratio. Defaults to 0.0.
        iternb Maximum number of iterations of NMF algorithm per run. Defaults to 2000.
        regu: Regularization target ('w', 'h', or 'both'). Defaults to 'both'.
        nruns: Number of runs to perform. Defaults to 100.
        random_seed: Random seed for reproducibility. Defaults to 0.
    """

    betaloss = "kullback-leibler" if divergence == 'kl' else 'frobenius'
    
    # Set regularization parameters
    alpha_W = alpha if regu in ['w', 'both'] else 0
    alpha_H = alpha if regu in ['h', 'both'] else 0

    print(f"Running NMF for {nsig} signatures...")

    best_evar = -np.inf
    best_cosine = -np.inf
    best_model = None
    best_H = None
    best_W = None

    cos_sim_list = []
    evar_list = []
    cos_sim_val_list = []
    evar_val_list = []

    for run in range(1, nruns+1):
        print(f"---- RUN #{run} ---")
        try:
            run_seed = random_seed + run

            model = NMF(
                n_components=nsig,
                init="nndsvdar",
                solver=solvernmf,
                beta_loss=betaloss,
                max_iter=iternb,
                random_state=run_seed,
                alpha_W=alpha_W,
                alpha_H=alpha_H,
                l1_ratio=l1ratio,
                verbose=False
            )
            W = model.fit_transform(X_train)   # samples x components
            H = model.components_              # components x features

            X_train_reco = W @ H
            evar = get_evar(X_train, X_train_reco)
            cosine = cosine_sim(X_train, X_train_reco)

            # Add W_val
            # Save H_run_j, W_train_run_j, W_val_run_j
            W_val = model.transform(X_val)
            X_val_reco = W_val @ H

            evar_val = get_evar(X_val, X_val_reco)
            cosine_val = cosine_sim(X_val, X_val_reco)

            # Save W and H per run
            pd.DataFrame(W, index=X_train.index).to_csv(f'{outdir}/W_train_run_{run}.csv')
            pd.DataFrame(W_val, index=X_val.index).to_csv(f'{outdir}/W_val_run_{run}.csv')
            pd.DataFrame(H, columns=X_train.columns).to_csv(f'{outdir}/H_run_{run}.csv')

            # Save metrics for each run
            cos_sim_list.append(cosine)
            evar_list.append(evar)
            cos_sim_val_list.append(cosine_val)
            evar_val_list.append(evar_val)

            if evar_val > best_evar and cosine_val > best_cosine:
                best_evar = evar_val
                best_cosine = cosine_val
                best_model = model
                best_run = run
                best_H = H
                best_W = W

            print(f"Complete Run {run}/{nruns} with evar: {evar_val:.4f}, cosine: {cosine_val:.4f}. Best evar: {best_evar:.4f}, best cosine: {best_cosine:.4f}")

        except Exception as e:
            print(f"Error during run {run}: {e}")
            continue
            
    if best_H is None or best_W is None:
        raise RuntimeError("All NMF runs failed. Check your input parameters or data.")


    save_metric_per_run(cos_sim_list, nruns, f'{outdir}/cossim_train.csv')
    save_metric_per_run(evar_list, nruns, f'{outdir}/evar_train.csv')
    save_metric_per_run(cos_sim_val_list, nruns, f'{outdir}/cossim_val.csv')
    save_metric_per_run(evar_val_list, nruns, f'{outdir}/evar_val.csv')

    print("NMF completed successfully.")
    print(f"Best run: {best_run} with evar: {best_evar:.4f}, cosine: {best_cosine:.4f}")

    return best_cosine, best_evar, best_H, best_W, best_model

In [ ]:
cos_sim_list = []
evar_list = []

for k in TOPICS:
    outdir = f'../results/{data_type}/{exp_group}/nmf_trained/nsig_{k}'
    os.makedirs(outdir, exist_ok=True)

    cosine, evar, H, W, best_model = run_nmf(
        X_train=X_train,
        X_val=X_val,
        nsig=k,
        outdir=outdir,
        alpha=0,
        l1ratio=0,
        divergence='kl',
        solvernmf="mu",
        nruns=50
    )

    W_test = best_model.transform(X_test)
    X_test_reco = W_test @ H
    
    evar_test = get_evar(X_test, X_test_reco)
    cosine_test = cosine_sim(X_test, X_test_reco)
    print(f"Test explained variance: {evar_test:.4f}")
    print(f"Test cosine similarity: {cosine_test:.4f}")

    # Save the results
    os.makedirs(f'{outdir}/best_run/', exist_ok=True)

    # For downstream analysis, I transforming matrices
    # W (features x k) <- H.T
    # H (k x samples)  <- W.T 
    pd.DataFrame(W.T, columns=X_train.index).to_csv(f'{outdir}/best_run/H_train_{k}.csv')
    pd.DataFrame(W_test.T, columns=X_test.index).to_csv(f'{outdir}/best_run/H_test_{k}.csv')
    pd.DataFrame(H.T, index=X_train.columns).to_csv(f'{outdir}/best_run/W_{k}.csv')

    cos_sim_list.append(cosine_test)
    evar_list.append(evar_test)

save_metric(cos_sim_list, f'{outdir}/cosine_similarities.csv')
save_metric(evar_list, f'{outdir}/explained_variance.csv')

Running NMF for 7 signatures...
----- RUN #1 ------
Complete Run 1/1 with evar: 0.8766, cosine: 0.9363. Best evar: 0.8766, best cosine: 0.9363
NMF completed successfully.
Best run: 1 with evar: 0.8766, cosine: 0.9363
Test explained variance: 0.9230
Test cosine similarity: 0.9607
